In [2]:
project_root_dir = !git rev-parse --show-toplevel
project_root_dir = project_root_dir[0]

import os
import sys
import numpy as np

sys.path.append(os.path.join(project_root_dir, 'lib'))
from math_utils import *

In [3]:
RNG = np.random.default_rng()

# Non batched average

In [4]:
for i in range(1, 100):
    a = RNG.uniform(size=i)
    f = RecursiveAverageFilter()
    mean = list(map(lambda x: f(x.item()), a))[-1]
    assert np.isclose(a.mean(), mean)

# Batched average

I'm writing Python code with numpy. I have following class:
```python
class RecursiveAverageFilter:
    def __init__(self):
        self.n = 0
        self.v = 0

    def __call__(self, x):
        self.n += 1
        alpha = (self.n - 1) / self.n
        self.v = alpha * self.v + (1 - alpha) * x
        return self.v
```
Currently it can compute average for each single incoming x. Suppose x is coming in batches, e.g. one hundred of the same x. In order to compute average I need to call this filter 100 times for the same x in cycle. And this is not great. How can I modify code so it computes average without cycles?

===

To update a recursive average with a batch of identical values without using a loop, you can treat the batch as a weighted addition to the current average.
Mathematically, if you have $n$ existing values with average $v$, and you add $k$ new instances of value $x$, the new average $v_{new}$ is:

$$v_{new} = \frac{n \cdot v + k \cdot x}{n + k}$$ 

**Updated Class Implementation**

```python
class RecursiveAverageFilter:
    def __init__(self):
        self.n = 0
        self.v = 0.0

    def __call__(self, x, batch_size=1):
        """
        Updates the average with batch_size instances of x.
        """
        n_new = self.n + batch_size
        # Calculate new average using weighted sum
        self.v = (self.n * self.v + batch_size * x) / n_new
        self.n = n_new
        return self.v
```

**Why this works:**

* Efficiency: Instead of $k$ iterations, it uses a single constant-time ($O(1)$) calculation.
* Numpy Compatibility: Since you mentioned using NumPy, this formula works even if x is a NumPy array (e.g., if you are averaging multiple sensors or image pixels simultaneously).
* Precision: This batch approach is numerically equivalent to the iterative version but avoids the cumulative floating-point drift that can sometimes occur in very long loops.

In [5]:
a = [ (10, 10), (10, 20), (10, 30), (40, 3) ]
arr_acc = np.array([])
f = RecursiveAverageFilter()

for a_i in a:
    value = a_i[0]
    batch_size = a_i[1]
    arr = np.full(batch_size, value)
    arr_acc = np.r_[arr_acc, arr]
    f(value, batch_size)

print(arr_acc.mean(), f.v)
assert np.isclose(arr_acc.mean(), f.v)

11.428571428571429 11.428571428571429
